# RuralHealth Vision image service (Google Colab)

Use this notebook when the computer running the Flask app does not have a usable CUDA GPU. It starts a temporary authenticated image endpoint on the Colab GPU. Keep this notebook session running while using image generation.

Before you start, review and accept the selected Stable Diffusion model's terms. In Colab, add these entries under **Secrets** and grant this notebook access: `NGROK_AUTHTOKEN` and `RH_IMAGE_TOKEN`. Use a long random value for `RH_IMAGE_TOKEN`. Do not share or commit either secret.

In [ ]:
!pip -q install diffusers accelerate safetensors pyngrok

## Upload the project service module

Run this cell and select `colab_image_service.py` from the project repository.

In [ ]:
from google.colab import files
uploaded = files.upload()
if 'colab_image_service.py' not in uploaded:
    raise RuntimeError('Please upload colab_image_service.py from the project repository.')

## Load Stable Diffusion on the Colab GPU

Change `MODEL_ID` only to a Diffusers-compatible model whose license and terms you have reviewed. The first run downloads model files into the Colab runtime.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Runtime > Change runtime type, then rerun this cell.')

MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, use_safetensors=True
).to('cuda')
print('Loaded', MODEL_ID, 'on', torch.cuda.get_device_name(0))

## Start the temporary authenticated endpoint

Run this cell last and leave it running. Copy the printed base URL (without `/generate`) and set it as `COLAB_IMAGE_API_URL` in the PowerShell window that runs Flask. Set `COLAB_IMAGE_API_TOKEN` to the same `RH_IMAGE_TOKEN` value. The tunnel URL changes when the Colab runtime restarts.

In [ ]:
import os
from google.colab import userdata

os.environ['NGROK_AUTHTOKEN'] = userdata.get('NGROK_AUTHTOKEN')
os.environ['RH_IMAGE_TOKEN'] = userdata.get('RH_IMAGE_TOKEN')
from colab_image_service import start_colab_image_service
server, tunnel = start_colab_image_service(pipe)